# 🚀 ItsNotAI v2 - 双头模型训练

**新特性:**
- ✅ 双头架构: 多分类 + 二分类，更准确的 Real/AI 判断
- ✅ 强数据增强: JPEG压缩、高斯噪声模拟真实场景
- ✅ Loss 加权: 提升真实照片识别率
- ✅ 新数据集: FLUX, SD3, DALL-E 3, Midjourney v6/v7

**运行前确保:** Runtime → Change runtime type → **A100 GPU** (或 T4)

## 1️⃣ 检查 GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2️⃣ 安装依赖

In [ ]:
!pip install -q torch torchvision transformers scikit-learn tqdm pillow datasets huggingface_hub timm

## 3️⃣ 克隆仓库

In [ ]:
import os
if not os.path.exists('/content/ItsNotAI-Model-Training'):
    !git clone https://github.com/wanikua/ItsNotAI-Model-Training.git
else:
    %cd /content/ItsNotAI-Model-Training
    !git pull

%cd /content/ItsNotAI-Model-Training

import sys
sys.path.insert(0, '/content/ItsNotAI-Model-Training')

print('✅ 仓库准备完成!')

## 4️⃣ 下载数据集

支持的数据集:
- **Defactify**: SD2.1, SDXL, SD3, DALL-E 3, Midjourney (96K)
- **OpenFake**: FLUX 1.0/1.1, Midjourney v6/v7, Imagen (可选)
- **COCO**: 真实照片 (可选)

In [ ]:
import os
from pathlib import Path
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm
import shutil

DATA_ROOT = Path('/content/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================
# 配置要下载的数据集
# ============================================
DOWNLOAD_DEFACTIFY = True   # 推荐: SD3, DALL-E 3, Midjourney
DOWNLOAD_COCO = True        # 推荐: 真实照片
DOWNLOAD_OPENFAKE = False   # 可选: FLUX (较大)

# 每个类别的样本数量限制 (None = 全部)
SAMPLES_PER_CLASS = 20000

print(f"数据目录: {DATA_ROOT}")
print(f"下载配置: Defactify={DOWNLOAD_DEFACTIFY}, COCO={DOWNLOAD_COCO}, OpenFake={DOWNLOAD_OPENFAKE}")

In [ ]:
# ============================================
# 下载 Defactify 数据集 (SD3, DALL-E 3, Midjourney)
# ============================================
if DOWNLOAD_DEFACTIFY:
    print("\n" + "="*60)
    print("📥 下载 Defactify 数据集...")
    print("="*60)
    
    defactify_dir = DATA_ROOT / 'artifact'
    defactify_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        ds = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset", split="train")
        
        # Label_B 映射: 0=Real, 1=SD21, 2=SDXL, 3=SD3, 4=DALLE3, 5=Midjourney
        source_names = ['Real', 'SD21', 'SDXL', 'SD3', 'DALLE3', 'Midjourney']
        
        # 统计每个类别的数量
        class_counts = {i: 0 for i in range(6)}
        
        for item in tqdm(ds, desc="处理 Defactify"):
            label_b = item['Label_B']
            
            # 检查是否达到限制
            if SAMPLES_PER_CLASS and class_counts[label_b] >= SAMPLES_PER_CLASS:
                continue
            
            source_name = source_names[label_b]
            source_dir = defactify_dir / source_name
            source_dir.mkdir(parents=True, exist_ok=True)
            
            # 保存图片
            img = item['Image']
            img_path = source_dir / f"{class_counts[label_b]:06d}.jpg"
            img.save(img_path, quality=95)
            
            class_counts[label_b] += 1
            
            # 检查是否所有类别都达到限制
            if SAMPLES_PER_CLASS and all(c >= SAMPLES_PER_CLASS for c in class_counts.values()):
                break
        
        print(f"\n✅ Defactify 下载完成!")
        for i, name in enumerate(source_names):
            print(f"   {name}: {class_counts[i]} 张")
            
    except Exception as e:
        print(f"❌ Defactify 下载失败: {e}")

In [ ]:
# ============================================
# 下载 COCO 数据集 (真实照片)
# ============================================
if DOWNLOAD_COCO:
    print("\n" + "="*60)
    print("📥 下载 COCO 真实照片...")
    print("="*60)
    
    coco_dir = DATA_ROOT / 'artifact' / 'COCO'
    coco_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        ds = load_dataset("HuggingFaceM4/COCO", split="train", streaming=True)
        
        count = 0
        max_samples = SAMPLES_PER_CLASS or 30000
        
        for item in tqdm(ds, desc="处理 COCO", total=max_samples):
            if count >= max_samples:
                break
            
            try:
                img = item['image']
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                img_path = coco_dir / f"{count:06d}.jpg"
                img.save(img_path, quality=95)
                count += 1
            except:
                continue
        
        print(f"\n✅ COCO 下载完成! {count} 张真实照片")
        
    except Exception as e:
        print(f"❌ COCO 下载失败: {e}")

In [ ]:
# ============================================
# 下载 OpenFake 数据集 (FLUX, 可选)
# ============================================
if DOWNLOAD_OPENFAKE:
    print("\n" + "="*60)
    print("📥 下载 OpenFake 数据集 (包含 FLUX)...")
    print("="*60)
    print("⚠️ 注意: OpenFake 数据集较大，下载可能需要较长时间")
    
    openfake_dir = DATA_ROOT / 'artifact' / 'FLUX'
    openfake_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # OpenFake 需要登录 HuggingFace
        # 如果没有 token，跳过
        ds = load_dataset("ComplexDataLab/OpenFake", split="train", streaming=True)
        
        count = 0
        max_samples = SAMPLES_PER_CLASS or 10000
        
        for item in tqdm(ds, desc="处理 OpenFake", total=max_samples):
            if count >= max_samples:
                break
            
            try:
                # 只下载 FLUX 生成的图片
                if 'flux' in str(item.get('generator', '')).lower():
                    img = item['image']
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    
                    img_path = openfake_dir / f"{count:06d}.jpg"
                    img.save(img_path, quality=95)
                    count += 1
            except:
                continue
        
        print(f"\n✅ OpenFake (FLUX) 下载完成! {count} 张")
        
    except Exception as e:
        print(f"❌ OpenFake 下载失败 (可能需要 HuggingFace 登录): {e}")

In [ ]:
# ============================================
# 创建 metadata.csv 文件
# ============================================
import csv

print("\n" + "="*60)
print("📝 创建 metadata.csv 文件...")
print("="*60)

artifact_dir = DATA_ROOT / 'artifact'
total_images = 0

for source_dir in artifact_dir.iterdir():
    if source_dir.is_dir():
        images = list(source_dir.glob('*.jpg')) + list(source_dir.glob('*.png'))
        
        if images:
            meta_path = source_dir / 'metadata.csv'
            with open(meta_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=['image_path', 'target', 'category'])
                writer.writeheader()
                
                for img_path in images:
                    # Real sources: Real, COCO, ImageNet, etc.
                    is_real = source_dir.name.lower() in ['real', 'coco', 'imagenet', 'ffhq', 'celebahq', 'lsun']
                    writer.writerow({
                        'image_path': img_path.name,
                        'target': 0 if is_real else 1,
                        'category': source_dir.name
                    })
            
            print(f"   {source_dir.name}: {len(images)} 张")
            total_images += len(images)

print(f"\n✅ 总计: {total_images} 张图片")

## 5️⃣ 开始训练! 🎯

**双头模式特性:**
- 多分类头: 识别具体来源 (SD3, DALL-E 3, Midjourney...)
- 二分类头: 直接判断 Real / AI
- Loss 加权: `[1.5, 1.0]` 提升真实照片识别率

In [ ]:
%cd /content/ItsNotAI-Model-Training

# 重载模块
import importlib
import src.dataset.combined_dataset
importlib.reload(src.dataset.combined_dataset)
import src.training.train_vit
importlib.reload(src.training.train_vit)
import src.training.config
importlib.reload(src.training.config)

from src.training.config import TrainingConfig
from src.training.train_vit import Trainer
from pathlib import Path

# ============================================
# 训练配置
# ============================================

# 使用双头模式配置
config = TrainingConfig.for_colab_dual_head()

# 数据配置
config.data_root = Path('/content/data')
config.output_dir = Path('/content/outputs')
config.limit = None  # None = 使用全部数据, 或设置数字限制

# 训练参数
config.batch_size = 64       # A100: 128, T4: 32-64
config.num_epochs = 10       # 推荐 8-15
config.learning_rate = 5e-6  # 双头模式推荐较小学习率

# 数据增强
config.strong_augmentation = True  # JPEG压缩、噪声等

# 二分类 Loss 权重 [Real, AI]
# 提高 Real 权重可以提升真实照片识别率
config.binary_class_weights = [1.5, 1.0]  # 默认
# config.binary_class_weights = [2.0, 1.0]  # 如果还是误判真实照片

# 其他
config.use_mlflow = False

print("\n" + "="*60)
print("📋 训练配置")
print("="*60)
print(f"模式: 双头 (多分类 + 二分类)")
print(f"数据目录: {config.data_root}")
print(f"Batch size: {config.batch_size}")
print(f"Epochs: {config.num_epochs}")
print(f"学习率: {config.learning_rate}")
print(f"强数据增强: {config.strong_augmentation}")
print(f"二分类权重: {config.binary_class_weights}")
print("="*60)

In [ ]:
# ============================================
# 开始训练
# ============================================
trainer = Trainer(config)
trainer.train()

## 6️⃣ 上传到 HuggingFace

In [ ]:
# ============================================
# 上传模型到 HuggingFace Hub
# ============================================
from huggingface_hub import HfApi, login

# 填入你的 HuggingFace Token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # 从 https://huggingface.co/settings/tokens 获取
MODEL_ID = "boluobobo/ItsNotAI-ai-detector-v2"  # 你的模型 ID

if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
    login(token=HF_TOKEN)
    
    api = HfApi()
    api.create_repo(repo_id=MODEL_ID, exist_ok=True)
    
    api.upload_folder(
        folder_path="/content/outputs/best_model",
        repo_id=MODEL_ID,
        repo_type="model",
    )
    
    print(f"\n✅ 模型已上传到: https://huggingface.co/{MODEL_ID}")
else:
    print("⚠️ 请填入你的 HuggingFace Token")

## 7️⃣ 保存到 Google Drive (备份)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/AI-Models"
!cp -r /content/outputs/best_model "/content/drive/MyDrive/AI-Models/ItsNotAI-v2"

print('\n✅ 模型已备份到 Google Drive: AI-Models/ItsNotAI-v2')

## 8️⃣ 测试模型

In [ ]:
from src.models.vit_detector import ViTDetector
from PIL import Image
import requests
from io import BytesIO

# 加载训练好的模型
model = ViTDetector.load('/content/outputs/best_model')

def test_image(img):
    """测试图片"""
    # 多分类预测
    result = model.predict_multiclass(img, top_k=5)
    
    # 二分类预测 (使用二分类头)
    real_prob, ai_prob = model.get_real_vs_fake_prob(img)
    
    print(f'\n{"="*50}')
    print(f'🎯 判断结果: {"✅ 真实照片" if real_prob > ai_prob else "❌ AI 生成"}')
    print(f'   Real: {real_prob:.1%} | AI: {ai_prob:.1%}')
    print(f'{"="*50}')
    print(f'\n📊 多分类详情:')
    print(f'   预测来源: {result.predicted_source}')
    print(f'   置信度: {result.confidence:.1%}')
    print(f'\n   Top 5 可能来源:')
    for source, prob in result.top_k:
        marker = "📷" if model.source_is_real.get(source, False) else "🤖"
        print(f'   {marker} {source}: {prob:.1%}')
    
    return result

print('模型加载完成! 使用 test_image(img) 测试图片')

In [ ]:
# 上传图片测试
from google.colab import files

print('📤 上传图片测试:')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'\n测试文件: {filename}')
    img = Image.open(filename).convert('RGB')
    test_image(img)

In [ ]:
# 测试 URL 图片
def test_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    return test_image(img)

# 示例:
# test_url('https://example.com/image.jpg')